# 04 — Market Regime Detection

**Purpose:** Identify distinct market regimes from volatility and momentum features.

We use unsupervised clustering to discover recurrent market states:
- Low volatility / trending
- High volatility / ranging
- Breakout / expansion  
- Mean-reversion / compression

Three algorithms are compared: **K-Means**, **GMM**, and **HDBSCAN**.

Regimes are then used to:
1. Condition strategy signals (only trade in high-probability regimes)
2. Analyse per-regime feature distributions
3. Diagnose strategy robustness across different market states

**Inputs:** `data/features/{symbol}_features.parquet`  
**Outputs:** `data/features/{symbol}_regimes.parquet`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils.config import load_config, get_symbol, get_paths
from src.models.regime import (
    prepare_regime_features, fit_kmeans, fit_gmm,
    kmeans_elbow, gmm_bic_sweep, interpret_regimes, assign_regime_labels
)

cfg    = load_config()
paths  = get_paths(cfg, "..")
symbol = get_symbol(cfg)

In [ ]:
feat_path = paths["features"] / f"{symbol}_features.parquet"
features  = pd.read_parquet(feat_path)
print(f"Features loaded: {features.shape}")

## 1. Prepare features for clustering

In [ ]:
X_scaled, scaler, _ = prepare_regime_features(features, use_pca=False)
print(f"Regime feature matrix: {X_scaled.shape}")

## 2. Determine optimal number of clusters

In [ ]:
inertias = kmeans_elbow(X_scaled, k_range=range(2, 9))
bic_df   = gmm_bic_sweep(X_scaled, n_range=range(2, 8))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(inertias.index, inertias.values, "o-", color="steelblue")
axes[0].set_title("K-Means Elbow")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].grid(alpha=0.3)

axes[1].plot(bic_df["n_components"], bic_df["bic"], "o-", label="BIC", color="navy")
axes[1].plot(bic_df["n_components"], bic_df["aic"], "s--", label="AIC", color="coral")
axes[1].set_title("GMM BIC / AIC")
axes[1].set_xlabel("n_components")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/regime_selection.png", dpi=120)
plt.show()

## 3. K-Means clustering

In [ ]:
N_CLUSTERS = 4  # adjust based on elbow plot above

km_labels, km_model = fit_kmeans(X_scaled, n_clusters=N_CLUSTERS)

# Assign back to original index
km_regime = assign_regime_labels(features, km_labels)

print("K-Means regime counts:")
print(km_regime.value_counts().sort_index())

In [ ]:
# Align labels to non-NaN rows for interpretation
from src.models.regime import _REGIME_FEATURES
cols   = [c for c in _REGIME_FEATURES if c in features.columns]
valid_features = features[cols].dropna()
valid_features["km_regime"] = km_labels

regime_summary = valid_features.groupby("km_regime")[cols].mean().round(3)
print("\nK-Means regime feature means:")
display(regime_summary)

## 4. Gaussian Mixture Model

In [ ]:
gmm_labels, gmm_model = fit_gmm(X_scaled, n_components=N_CLUSTERS)
gmm_regime = assign_regime_labels(features, gmm_labels)

print("GMM regime counts:")
print(gmm_regime.value_counts().sort_index())

## 5. HDBSCAN (optional)

In [ ]:
try:
    from src.models.regime import fit_hdbscan
    # Subsample for speed — HDBSCAN is O(n^2) in memory for large datasets
    sample_size = min(50_000, len(X_scaled))
    idx_sample  = np.random.choice(len(X_scaled), sample_size, replace=False)
    X_sample    = X_scaled[idx_sample]

    hdb_labels, _ = fit_hdbscan(X_sample, min_cluster_size=200, min_samples=20)
    print(f"HDBSCAN clusters: {len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)}")
    print(f"Noise: {(hdb_labels==-1).sum()} ({(hdb_labels==-1).mean()*100:.1f}%)")
except ImportError:
    print("hdbscan not installed. Run: pip install hdbscan")

## 6. Save regimes and visualise time series

In [ ]:
# Save K-Means regimes as the primary regime assignment
regime_df = pd.DataFrame({
    "km_regime":  km_regime,
    "gmm_regime": gmm_regime,
}, index=features.index)

out_path = paths["features"] / f"{symbol}_regimes.parquet"
regime_df.to_parquet(out_path, engine="pyarrow")
print(f"Regimes saved → {out_path}")

In [ ]:
# Plot price coloured by regime (last 2000 bars)
n_plot = 2000
df_plot = features[["close"]].iloc[-n_plot:].copy()
df_plot["regime"] = km_regime.iloc[-n_plot:]

colors = {0: "steelblue", 1: "green", 2: "red", 3: "orange", -1: "grey"}

fig, ax = plt.subplots(figsize=(16, 5))
for reg in df_plot["regime"].dropna().unique():
    mask = df_plot["regime"] == reg
    ax.scatter(
        df_plot.index[mask],
        df_plot["close"][mask],
        s=1, color=colors.get(int(reg), "purple"),
        label=f"Regime {int(reg)}"
    )
ax.set_title(f"{symbol} — Last {n_plot} bars coloured by K-Means Regime")
ax.legend(loc="upper left", markerscale=5)
plt.tight_layout()
plt.savefig("../reports/regime_timeseries.png", dpi=120)
plt.show()